# B0 baseline — без Pareto, без RL, 3 random seed

Этот notebook запускает **базовую стратегию B0** для задачи с непересекающейся разметкой.

### Что важно
- **Pareto НЕ используется**.
- **REINVENT / RL НЕ используются**.
- Генерация: стохастическая рекомбинация и мутация фрагментов из $D_A$ и $D_B$.
- **7 evaluator-моделей** используются только для surrogate-scoring.
- **7 независимых oracle-моделей** используются только для финальной оценки.
- Запуски: **seed = 42, 101, 2024**.
- По каждому seed генерируется **1000 уникальных валидных молекул**.
- Итог: 3000 кандидатов + таблица `mean ± std`.

Это корректный **B0 baseline**, с которым затем сравнивается M1 (Pareto-RL).

In [ ]:
# Проверка среды
import os, sys, platform, subprocess, json, time
print('Python:', sys.version)
print('Platform:', platform.platform())
print('Working dir:', os.getcwd())
print('GPU для B0 не требуется.')

## 1. Клонируем Case и фиксируем версию

Используется зафиксированный commit, чтобы B0 был воспроизводимым.

In [ ]:
import os, shutil, subprocess

CASE_DIR = '/content/Case'
CASE_COMMIT = 'aa9c32fb26a39e518e420bb333298a20d59cbd94'

if os.path.exists(CASE_DIR):
    shutil.rmtree(CASE_DIR)

subprocess.run(['git','clone','--quiet','https://github.com/suharevalexey/Case.git', CASE_DIR], check=True)
subprocess.run(['git','-C',CASE_DIR,'checkout','--quiet', CASE_COMMIT], check=True)
print('Case commit:', subprocess.check_output(['git','-C',CASE_DIR,'rev-parse','HEAD'], text=True).strip())

## 2. Устанавливаем зависимости B0

Здесь не ставится REINVENT4: baseline B0 — отдельная стохастическая графовая стратегия.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{CASE_DIR}/requirements.txt'], check=True)
print('Dependencies installed.')

## 3. Проверяем, что B0 действительно без Pareto

Стратегия должна содержать только генерацию через crossover/mutation и последующую оценку evaluator/oracle.

In [ ]:
from pathlib import Path
src = Path(CASE_DIR, 'src', 'strategy_b0.py').read_text(encoding='utf-8')

print('Есть BaselineGeneratorB0:', 'class BaselineGeneratorB0' in src)
print('Есть stochastic generation:', 'generate_candidate' in src)
print('Есть 3-seed runner:', 'run_experiment_b0' in src)
print('Слово Pareto в strategy_b0.py:', 'pareto' in src.lower())

assert 'class BaselineGeneratorB0' in src
assert 'run_experiment_b0' in src
assert 'pareto' not in src.lower(), 'В B0 найдено упоминание Pareto — проверьте источник.'
print('\nOK: B0 не использует Pareto.')

## 4. Проверяем evaluator + oracle

Для каждой молекулы рассчитываются все 7 surrogate-свойств, а затем независимые oracle-предсказания. Oracle **не влияет на генерацию**.

In [ ]:
import sys, os
sys.path.insert(0, f'{CASE_DIR}/src')
from evaluators import MODELS_META

print('Количество свойств:', len(MODELS_META))
for k, meta in MODELS_META.items():
    print(f"{meta['group']} | {k:40s} | [{meta['constraint_min']}, {meta['constraint_max']}] {meta['unit']}")
assert len(MODELS_META) == 7

## 5. Параметры B0

Одинаковый размер итоговой выборки для каждого seed: **1000 уникальных молекул**.

In [ ]:
SEEDS = [42, 101, 2024]
N_SAMPLES_PER_SEED = 1000
print('SEEDS =', SEEDS)
print('N per seed =', N_SAMPLES_PER_SEED)
print('Total final candidates =', len(SEEDS) * N_SAMPLES_PER_SEED)

## 6. Запуск B0

Полный вывод пишется в лог-файл, чтобы не перегружать websocket Colab.

In [ ]:
import os, subprocess, time
from pathlib import Path

LOG_DIR = Path('/content/b0_logs')
LOG_DIR.mkdir(exist_ok=True)
LOG_PATH = LOG_DIR / 'b0_3seeds.log'

cmd = [
    sys.executable,
    f'{CASE_DIR}/src/strategy_b0.py',
    '--n_samples', str(N_SAMPLES_PER_SEED),
    '--seeds', *map(str, SEEDS)
]

print('Command:', ' '.join(cmd))
print('Full log:', LOG_PATH)

t0 = time.time()
with open(LOG_PATH, 'w', encoding='utf-8') as log:
    p = subprocess.Popen(cmd, cwd=CASE_DIR, stdout=log, stderr=subprocess.STDOUT, text=True)
    while p.poll() is None:
        elapsed = (time.time() - t0)/60
        print(f'[{elapsed:.1f} min] B0 running...')
        time.sleep(30)
    rc = p.returncode

print('Return code:', rc)
print('Elapsed min:', round((time.time()-t0)/60, 2))

lines = LOG_PATH.read_text(encoding='utf-8', errors='replace').splitlines()
print('\n--- LAST LOG LINES ---')
print('\n'.join(lines[-160:] if rc != 0 else lines[-40:]))
if rc != 0:
    raise RuntimeError(f'B0 failed, see {LOG_PATH}')

## 7. Загружаем результаты

In [ ]:
import pandas as pd, json, numpy as np
from pathlib import Path

RES_DIR = Path(CASE_DIR) / 'results'
generated_path = RES_DIR / 'generated_B0.csv'
summary_path = RES_DIR / 'metrics_summary_B0.csv'
metrics_json_path = RES_DIR / 'metrics_B0.json'

assert generated_path.exists(), generated_path
assert summary_path.exists(), summary_path
assert metrics_json_path.exists(), metrics_json_path

generated = pd.read_csv(generated_path)
summary = pd.read_csv(summary_path)
metrics = json.loads(metrics_json_path.read_text(encoding='utf-8'))

print('generated rows:', len(generated))
print('seeds:', sorted(generated['seed'].dropna().unique().tolist()))
print('unique SMILES total:', generated['SMILES'].nunique())
display(summary)

## 8. Ключевые метрики по seed

In [ ]:
seed_rows = summary[summary['method'] == 'B0'].copy()
cols = [
    'seed','Validity','Uniqueness','Novelty','Internal_Diversity',
    'JSR_Oracle (Joint Success Rate)','JSR_Surrogate','Overall_Pass_Constraints',
    'SAScore_Mean','AD_Fraction_in_Both_AD'
]
display(seed_rows[cols])

## 9. Итог mean ± std

In [ ]:
agg = metrics['aggregated']
keys = [
    'Validity','Uniqueness','Novelty','Internal_Diversity',
    'JSR_Oracle (Joint Success Rate)','JSR_Surrogate','Overall_Pass_Constraints',
    'SAScore_Mean','AD_Fraction_in_Both_AD'
]
rows=[]
for k in keys:
    mean = agg.get(f'{k}_mean', np.nan)
    std = agg.get(f'{k}_std', np.nan)
    rows.append({'metric': k, 'mean': mean, 'std': std, 'mean ± std': f'{mean:.4f} ± {std:.4f}'})
agg_table = pd.DataFrame(rows)
display(agg_table)

## 10. Графики B0

In [ ]:
import matplotlib.pyplot as plt

plot_df = seed_rows[['seed','JSR_Oracle (Joint Success Rate)','JSR_Surrogate']].set_index('seed')
ax = plot_df.plot(kind='bar', figsize=(8,4))
ax.set_ylabel('Rate')
ax.set_title('B0: Oracle JSR vs Surrogate JSR')
ax.set_ylim(0, max(0.4, plot_df.to_numpy().max()*1.2))
plt.tight_layout()
plt.show()

In [ ]:
plot_df2 = seed_rows[['seed','Validity','Novelty','Internal_Diversity','AD_Fraction_in_Both_AD']].set_index('seed')
ax = plot_df2.plot(kind='bar', figsize=(10,4))
ax.set_ylabel('Rate')
ax.set_title('B0 quality metrics by seed')
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

## 11. Проверка generated_B0.csv

In [ ]:
required_cols = [
    'SMILES','method_id','seed','novelty','dist_to_D_A','dist_to_D_B',
    'synthetic_accessibility','pass_surrogate_all','pass_oracle_all','pass_constraints'
]
missing = [c for c in required_cols if c not in generated.columns]
print('Missing required columns:', missing)
assert not missing

print('\nColumns:', len(generated.columns))
print(generated.columns.tolist())
display(generated.head())

## 12. Failure analysis B0

Показываем, где surrogate и oracle расходятся.

In [ ]:
g = generated.copy()
g['false_positive_surrogate'] = ((g['pass_surrogate_all'] == 1) & (g['pass_oracle_all'] == 0)).astype(int)
g['false_negative_surrogate'] = ((g['pass_surrogate_all'] == 0) & (g['pass_oracle_all'] == 1)).astype(int)

fail_summary = g.groupby('seed').agg(
    n=('SMILES','size'),
    surrogate_pass=('pass_surrogate_all','sum'),
    oracle_pass=('pass_oracle_all','sum'),
    false_positive=('false_positive_surrogate','sum'),
    false_negative=('false_negative_surrogate','sum'),
).reset_index()

display(fail_summary)

## 13. Готовая формулировка для защиты

In [ ]:
jsr_m = agg['JSR_Oracle (Joint Success Rate)_mean']
jsr_s = agg['JSR_Oracle (Joint Success Rate)_std']
nov_m = agg['Novelty_mean']
nov_s = agg['Novelty_std']
div_m = agg['Internal_Diversity_mean']
div_s = agg['Internal_Diversity_std']
ad_m = agg['AD_Fraction_in_Both_AD_mean']
ad_s = agg['AD_Fraction_in_Both_AD_std']

lines = [
    'B0 — это независимый baseline без Pareto и без reinforcement learning.',
    'Он генерирует молекулы стохастической рекомбинацией и мутацией фрагментов D_A и D_B.',
    'Мы выполнили 3 независимых запуска с seed 42, 101 и 2024, по 1000 уникальных кандидатов на запуск.',
    f'Средний независимый Oracle JSR составил {jsr_m*100:.2f}% ± {jsr_s*100:.2f}%, novelty — {nov_m*100:.2f}% ± {nov_s*100:.2f}%, internal diversity — {div_m:.4f} ± {div_s:.4f}, а доля молекул внутри обоих applicability domains — {ad_m*100:.2f}% ± {ad_s*100:.2f}%.',
    'Эта стратегия используется как B0 для честного сравнения с M1 при одинаковом протоколе независимой oracle-оценки.'
]
print('\n'.join(lines))

## 14. Экспорт результатов B0

In [ ]:
import shutil, os
from pathlib import Path

OUT_DIR = Path('/content/B0_NO_PARETO_RESULTS')
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir()

for src in [generated_path, summary_path, metrics_json_path, LOG_PATH]:
    shutil.copy2(src, OUT_DIR / src.name)

agg_table.to_csv(OUT_DIR / 'B0_mean_std_key_metrics.csv', index=False)
fail_summary.to_csv(OUT_DIR / 'B0_failure_analysis.csv', index=False)

zip_base = '/content/B0_NO_PARETO_3SEEDS_RESULTS'
shutil.make_archive(zip_base, 'zip', OUT_DIR)
print('Saved:', zip_base + '.zip')
print('Folder:', OUT_DIR)

---
### Итог

**B0 = baseline без Pareto.**  
Он ничего не оптимизирует по Парето и не обучает генератор через RL. Это специально простой, интерпретируемый и воспроизводимый baseline для сравнения с M1.